# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [2]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [3]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [4]:
# 🛠️ TOOL 3 (Bonus): Word Counter

def word_counter(text: str) -> dict:
    """Count words and characters in a piece of text."""
    try:
        words = text.split()
        return {
            "word_count": len(words),
            "char_count": len(text)
        }
    except Exception:
        return {"word_count": 0, "char_count": 0}

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- If query contains "count" → use word counter (bonus tool)
- Else → general response

In [5]:
# 📝 Logging Setup (Bonus)

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("agent")

In [6]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

import re

def agent(query: str):
    """
    Single-agent router that:
      1. Validates the incoming query
      2. Routes it to the correct tool based on keyword pattern checks
      3. Returns a structured JSON-style response: {"type": ..., "result": ...}
    """

    # --- Step 1: Basic validation ---
    if not isinstance(query, str) or query.strip() == "":
        logger.error("Empty or invalid query received.")
        return {"type": "error", "result": "Query must be a non-empty string"}

    query_lower = query.lower()

    try:
        # --- Route 1: Calculator ---
        if "calculate" in query_lower:
            logger.info(f"Routing to Calculator Tool | query='{query}'")

            match = re.search(r"calculate\s+(.*)", query_lower)
            expression = match.group(1).strip() if match else ""
            # keep only characters valid in a basic math expression
            expression = re.sub(r"[^0-9+\-*/().\s]", "", expression).strip()

            if not expression:
                logger.error("No valid mathematical expression found in query.")
                return {"type": "error", "result": "No valid mathematical expression found"}

            result = calculator(expression)
            if result == "Error in calculation":
                logger.error(f"Calculator failed on expression: '{expression}'")
                return {"type": "error", "result": result}

            return {"type": "calculation", "result": result}

        # --- Route 2: Keyword Extractor ---
        elif "keywords" in query_lower:
            logger.info(f"Routing to Keyword Extractor Tool | query='{query}'")

            match = re.search(r"keywords\s+(?:from|in|of)?\s*(.*)", query, flags=re.IGNORECASE)
            text = match.group(1).strip() if match and match.group(1).strip() else query

            keywords = extract_keywords(text)
            return {"type": "keywords", "result": keywords}

        # --- Route 3: Word Counter (bonus tool) ---
        elif "count" in query_lower:
            logger.info(f"Routing to Word Counter Tool | query='{query}'")

            match = re.search(r"count\s+(?:words\s+in|words\s+of)?\s*(.*)", query, flags=re.IGNORECASE)
            text = match.group(1).strip() if match and match.group(1).strip() else query

            counts = word_counter(text)
            return {"type": "count", "result": counts}

        # --- Route 4: General fallback ---
        else:
            logger.info(f"Routing to General Response | query='{query}'")
            general_response = (
                f"I received your query: '{query}'. "
                "This looks like a general question, so no specialized tool was used."
            )
            return {"type": "general", "result": general_response}

    except Exception as e:
        logger.error(f"Unexpected error while processing query '{query}': {e}")
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [7]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Count words in The quick brown fox jumps over the lazy dog",
    "Calculate 10 / 0",
    ""
]

for q in queries:
    print("Query:", repr(q))
    print("Response:", agent(q))
    print("-" * 50)

2026-08-09 22:35:19,528 - INFO - Routing to Calculator Tool | query='Calculate 20 + 5'
2026-08-09 22:35:19,528 - INFO - Routing to Keyword Extractor Tool | query='Extract keywords from Artificial Intelligence is transforming industries'
2026-08-09 22:35:19,528 - INFO - Routing to General Response | query='What is machine learning?'
2026-08-09 22:35:19,535 - INFO - Routing to Word Counter Tool | query='Count words in The quick brown fox jumps over the lazy dog'
2026-08-09 22:35:19,535 - INFO - Routing to Calculator Tool | query='Calculate 10 / 0'
2026-08-09 22:35:19,535 - ERROR - Calculator failed on expression: '10 / 0'
2026-08-09 22:35:19,535 - ERROR - Empty or invalid query received.


Query: 'Calculate 20 + 5'
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: 'Extract keywords from Artificial Intelligence is transforming industries'
Response: {'type': 'keywords', 'result': ['transforming', 'industries', 'intelligence', 'artificial']}
--------------------------------------------------
Query: 'What is machine learning?'
Response: {'type': 'general', 'result': "I received your query: 'What is machine learning?'. This looks like a general question, so no specialized tool was used."}
--------------------------------------------------
Query: 'Count words in The quick brown fox jumps over the lazy dog'
Response: {'type': 'count', 'result': {'word_count': 9, 'char_count': 43}}
--------------------------------------------------
Query: 'Calculate 10 / 0'
Response: {'type': 'error', 'result': 'Error in calculation'}
--------------------------------------------------
Query: ''
Response: {'type': 'error', 'result': 'Query

In [8]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Query:", user_input)
    print("Response:", agent(user_input))

2026-08-09 22:35:27,380 - INFO - Routing to Calculator Tool | query='calculate 10+2 '


Query: calculate 10+2 
Response: {'type': 'calculation', 'result': '12'}


2026-08-09 22:35:35,677 - INFO - Routing to General Response | query='what is machine learning?'


Query: what is machine learning?
Response: {'type': 'general', 'result': "I received your query: 'what is machine learning?'. This looks like a general question, so no specialized tool was used."}


2026-08-09 22:35:46,616 - INFO - Routing to Word Counter Tool | query='count words in shubham dogra'


Query: count words in shubham dogra
Response: {'type': 'count', 'result': {'word_count': 2, 'char_count': 13}}
